# Phase 1 - Importing Dataset and Preprocessing Dataset
First checking available device and selecting accordingly

In [ ]:
import torch
# Select GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Div2K and Flickr2K image datasets are used

In [ ]:
import os

INPUT_DIR = "/kaggle/input"

for root, dirs, files in os.walk(INPUT_DIR):
    print(f"{root} -> {len(files)} files")

In [ ]:
import os
import random
from glob import glob

import numpy as np
from PIL import Image

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

## Combining and Collecting Dataset Paths
Combining the two dataset images

In [ ]:
DIV2K_TRAIN = "/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR"

DIV2K_VALID = "/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_valid_HR/DIV2K_valid_HR"

FLICKR2K = "/kaggle/input/datasets/daehoyang/flickr2k/Flickr2K"

In [ ]:
image_extensions = ("*.png", "*.jpg", "*.jpeg", "*.bmp")

train_image_paths = []
test_image_paths = []

for ext in image_extensions:
    train_image_paths.extend(glob(os.path.join(DIV2K_TRAIN, ext)))
    train_image_paths.extend(glob(os.path.join(FLICKR2K, ext)))
train_image_paths = sorted(train_image_paths)
print(f"Train Images: {len(train_image_paths)}")

for ext in image_extensions:
    test_image_paths.extend(glob(os.path.join(DIV2K_VALID, ext)))
test_image_paths = sorted(test_image_paths)
print(f"Test Images: {len(test_image_paths)}")

print(f"Total Images: {len(train_image_paths) + len(test_image_paths)}")

## Preprocessing Images

### Shuffling

In [ ]:
random.seed(42)
random.shuffle(train_image_paths)  #shuffle image paths
print(train_image_paths[:5])  #check

### Dynamic Preprocessing and Creating Dataset

#### Dynamic Class

In [ ]:
class SuperResolutionDataset(Dataset):
    
    def __init__(
        self,
        image_paths,
        hr_size=256,
        scale=2,
        training=True,
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    ):
        """
        Args:
            image_paths (list): List of HR image paths.
            hr_size (int): HR patch size used during training.
            scale (int): Upscaling factor (2x).
            training (bool): True for training, False for validation/testing.
            mean (tuple): Normalization mean.
            std (tuple): Normalization std.
        """

        self.image_paths = image_paths
        self.hr_size = hr_size
        self.scale = scale
        self.lr_size = hr_size // scale
        self.training = training
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.image_paths)

    def preprocess(self, image):
        # =====================================================
        # TRAINING
        # =====================================================
        if self.training:
            # Resize only if image is smaller than crop size
            if image.width < self.hr_size or image.height < self.hr_size:
                image = TF.resize(
                    image,
                    (
                        max(self.hr_size, image.height),
                        max(self.hr_size, image.width)
                    ),
                    interpolation=InterpolationMode.BICUBIC
                )

            # ---------------- Random Crop ----------------
            i, j, h, w = transforms.RandomCrop.get_params(
                image,
                output_size=(self.hr_size, self.hr_size)
            )
            hr = TF.crop(image, i, j, h, w)

            # ---------------- Horizontal Flip ----------------
            if random.random() < 0.5:
                hr = TF.hflip(hr)

            # ---------------- Vertical Flip ----------------
            if random.random() < 0.5:
                hr = TF.vflip(hr)

            # ---------------- Random Rotation By Random Angle ----------------
            if random.random() < 0.5:
                angle = random.choice([90, 180, 270])
                hr = TF.rotate(hr, angle)

        # =====================================================
        # TESTING (no cropping or augmentation)
        # =====================================================
        else:
            # Make dimensions divisible by scale factor
            width = image.width - (image.width % self.scale)
            height = image.height - (image.height % self.scale)

            hr = TF.crop(
                image,
                top=0,
                left=0,
                height=height,
                width=width
            )

        # =====================================================
        # CREATE LOW RESOLUTION IMAGE
        # =====================================================
        lr = TF.resize(
            hr,
            size=(
                hr.height // self.scale,
                hr.width // self.scale
            ),
            interpolation=InterpolationMode.BICUBIC,
            antialias=True
        )

        # =====================================================
        # TO TENSOR (0-255)
        # =====================================================

        hr = TF.to_tensor(hr)
        lr = TF.to_tensor(lr)

        # =====================================================
        # NORMALIZATION [-1,1]
        # =====================================================
        hr = TF.normalize(
            hr,
            self.mean,
            self.std
        )

        lr = TF.normalize(
            lr,
            self.mean,
            self.std
        )

        return lr, hr

    # get image and preprocess
    def __getitem__(self, index):
        image = Image.open(
            self.image_paths[index]
        ).convert("RGB")

        lr, hr = self.preprocess(image)

        return lr, hr

#### Creating Dataset

In [ ]:
train_dataset = SuperResolutionDataset(
    image_paths=train_image_paths,
    training=True
)

test_dataset = SuperResolutionDataset(
    image_paths=test_image_paths,
    training=False
)

print(f"Training Images : {len(train_dataset)}")
print(f"Validation Images : {len(test_dataset)}")

#### DataLoader

In [ ]:
BATCH_SIZE = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2, #allow processing in parallel
    pin_memory=True, #speeds up CPU to GPU transfers
    drop_last=True #keeps batches the same size 
)

val_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

### Verify DataLoader

In [ ]:
lr, hr = next(iter(train_loader))

print("Train LR :", lr.shape)
print("Train HR :", hr.shape)

In [ ]:
lr, hr = next(iter(val_loader))

print("Validation LR :", lr.shape)
print("Validation HR :", hr.shape)

#### Verify Sample

In [ ]:
import matplotlib.pyplot as plt

def denormalize(img):
    return img * 0.5 + 0.5

lr, hr = train_dataset[0]

lr = denormalize(lr).permute(1, 2, 0).numpy()
hr = denormalize(hr).permute(1, 2, 0).numpy()

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(lr)
plt.title("Low Resolution")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(hr)
plt.title("High Resolution")
plt.axis("off")

plt.show()

# Phase 2 - Generator
The generator learns to reconstruct missing details (edges, textures, fine patterns etc) in low res image.
It learns the difference between input and desired output (particularly in Residual Blocks).


Initial FE 
    **|**
Residual Blocks (16)
    **|**
Global Skip Connection
    **|**
Upsampling Blocks
    **|**
Output Layer

## Residual Block
Input -> Conv -> BatchNorm -> PReLU -> Conv -> BatchNorm -> Output

In [ ]:
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    """
    Standard SRGAN Residual Block
    
    Structure:
        Conv
        BatchNorm
        PReLU
        Conv
        BatchNorm
        Skip Connection
    """

    def __init__(self, channels=64):
        super().__init__()

        self.block = nn.Sequential(

            # First convolution
            nn.Conv2d(
                in_channels=channels,
                out_channels=channels,
                kernel_size=3, #3x3 kernel size
                stride=1, #no reduction in image res
                padding=1 #keeps original
            ),

            # Batch Norm
            nn.BatchNorm2d(channels), #normalizes feature maps

            #PReLU
            nn.PReLU(), #keeps negative values (-ve x alpha)

            # Second convolution
            nn.Conv2d(
                in_channels=channels,
                out_channels=channels,
                kernel_size=3,
                stride=1,
                padding=1
            ),

            # BatchNorm
            nn.BatchNorm2d(channels)
        )

    def forward(self, x):

        # Save original input
        residual = x

        # Pass through convolution block
        out = self.block(x)

        # Add skip connection
        out = out + residual

        return out

## Pixel Shuffle Block
Instead of Transpose Conv, SRGAN uses Pixel Shuffle which rearranges channels into spatial resolution. Here, it doubles the resolution of image using the learned features.

Conv -> PixelShuffle -> PReLU

In [ ]:
class PixelShuffleBlock(nn.Module):
    """
    Upsampling block using PixelShuffle

    Structure:
        Conv
        PixelShuffle
        PReLU
    """

    def __init__(self, channels=64, upscale_factor=2):
        super().__init__()

        self.block = nn.Sequential(

            # Increase channel count
            nn.Conv2d(
                in_channels=channels,
                out_channels=channels * (upscale_factor ** 2),
                kernel_size=3,
                stride=1,
                padding=1
            ),

            # Rearrange channels into pixels
            nn.PixelShuffle(upscale_factor),

            # Activation
            nn.PReLU()
        )

    def forward(self, x):
        return self.block(x)

## Arranging the SRGAN

In [ ]:
class Generator(nn.Module):
    def __init__(
        self,
        in_channels=3,
        num_channels=64,
        num_residual_blocks=16,
        upscale_factor=2
    ):
        super().__init__()

        # -----------------------------------------
        # Initial Feature Extraction
        # Convert RGB image into feature maps
        # -----------------------------------------
        self.initial = nn.Sequential(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=num_channels,
                kernel_size=9, #9x9 kernel (capture large contexts, original SRGAN paper)
                stride=1,
                padding=4
            ),
            nn.PReLU() #activation
        )

        # -----------------------------------------
        # Deep Feature Extraction
        # 16 Residual Blocks
        # -----------------------------------------
        self.residual_blocks = nn.Sequential(
            *[
                ResidualBlock(num_channels)
                for _ in range(num_residual_blocks)
            ]
        )

        # -----------------------------------------
        # Feature Fusion Layer
        # Combines all learned residual features
        # -----------------------------------------
        self.conv_block = nn.Sequential(
            nn.Conv2d(
                in_channels=num_channels,
                out_channels=num_channels,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.BatchNorm2d(num_channels)
        )

        # -----------------------------------------
        # Upsampling block
        # Increases resolution by 2×
        # -----------------------------------------
        self.upsample = PixelShuffleBlock(
            channels=num_channels,
            upscale_factor=upscale_factor
        )

        # -----------------------------------------
        # Output Layer
        # Converts feature maps back to RGB image
        # -----------------------------------------
        self.output = nn.Conv2d(
            in_channels=num_channels,
            out_channels=in_channels,
            kernel_size=9, #9x9, original SRGAN
            stride=1,
            padding=4
        )


    def forward(self, x):
        # -----------------------------------------
        # Initial feature extraction
        # -----------------------------------------
        initial_features = self.initial(x)

        # Save features for the global skip connection
        skip_connection = initial_features

        # -----------------------------------------
        # Deep feature learning
        # -----------------------------------------
        x = self.residual_blocks(initial_features)

        # -----------------------------------------
        # Feature fusion
        # -----------------------------------------
        x = self.conv_block(x)

        # -----------------------------------------
        # Global Skip Connection
        # -----------------------------------------
        x = x + skip_connection

        # -----------------------------------------
        # Increase image resolution
        # -----------------------------------------
        x = self.upsample(x)

        # -----------------------------------------
        # Convert feature maps into RGB image
        # -----------------------------------------
        x = self.output(x)
        x = torch.tanh(x)
        return x

## Testing generator

In [ ]:
generator = Generator().to(device)
lr = torch.randn(1, 3, 96, 96, device=device)
hr = generator(lr)
print(hr.shape)

# Phase 3 - Discriminator
Discriminator forces the Generator to generate finer details. It is a CNN classifier that reduces image size while increasing feature map depth.

## Discriminator Block

In [ ]:
import torch
import torch.nn as nn

class DiscriminatorBlock(nn.Module):
    """
    Basic convolutional block used in the SRGAN discriminator

    Structure:
        Conv2D -> (BatchNorm) -> LeakyReLU

    The first block in the discriminator does not use BatchNorm,
    following the original SRGAN paper.
    """
    
    def __init__(self, in_channels, out_channels, stride, use_bn=True):
        super().__init__()

        layers = [
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=stride,
                padding=1
            )
        ]

        if use_bn:
            # batch normalization
            layers.append(nn.BatchNorm2d(out_channels))

        layers.append(
            # Leaky relu keeps negative values (keeps neurons from dying)
            nn.LeakyReLU(negative_slope=0.2, inplace=True)
        )

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

## Arranging discriminator

In [ ]:
class Discriminator(nn.Module):
    """
    SRGAN Discriminator

    Input:
        High-resolution image (3 × 256 × 256)

    Output:
        Probability that the image is real.
    """

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            # 96x96 -> 96x96
            DiscriminatorBlock(
                3,  # 3 input channel
                64,  # 64 feature maps
                stride=1,  #keeps same dims
                use_bn=False
            ),

            # 96x96 -> 48x48
            DiscriminatorBlock(
                64,
                64,
                stride=2  #halves the dims
            ),

            # 48x48 -> 48x48
            DiscriminatorBlock(
                64,
                128,
                stride=1
            ),

            # 48x48 -> 24x24
            DiscriminatorBlock(
                128,
                128,
                stride=2
            ),

            # 24x24 -> 24x24
            DiscriminatorBlock(
                128,
                256,
                stride=1
            ),

            # 24x24 -> 12x12
            DiscriminatorBlock(
                256,
                256,
                stride=2
            ),

            # 12x12 -> 12x12
            DiscriminatorBlock(
                256,
                512,
                stride=1
            ),

            # 12x12 -> 6x6
            DiscriminatorBlock(
                512,
                512,
                stride=2
            )
        )

        self.classifier = nn.Sequential(
            # Convert feature maps into a 1D vector
            # shape becomes from (4,512,16,16) to (4,131072) - (batch,channels,height,width)
            nn.Flatten(),

            # Fully connected layer
            # 131072 to 1024
            nn.Linear(512 * 16 * 16, 1024),

            # +ve values -> x, -ve values -> 0.2x
            nn.LeakyReLU(0.2, inplace=True),

            # Output a single score
            nn.Linear(1024, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

## Testing discriminator

In [ ]:
# Create the discriminator
discriminator = Discriminator().to(device)
# Test with a batch of 4 HR images
dummy_input = torch.randn(4, 3, 256, 256).to(device)
output = discriminator(dummy_input)
print("Output shape:", output.shape)
print(output)

# Phase 4 - VGG Feature Extractor and Loss Calculation

## Pre-trained VGG as FE
Keeping the initial 36 layers of pre-trained CNN as done commonly.

In [ ]:
from torchvision.models import vgg19, VGG19_Weights
import torch
import torch.nn as nn

class VGGFeatureExtractor(nn.Module):
    """
    Extracts high-level image features using a pretrained VGG19 network

    These features are used to compute the perceptual/content loss
    between the generated SR image and the ground-truth HR image.
    """

    def __init__(self):
        super(VGGFeatureExtractor, self).__init__()

        # Load pretrained VGG19 model
        vgg = vgg19(weights=VGG19_Weights.IMAGENET1K_V1)

        # Keep layers up to conv5_4 (before maxpool5)
        self.feature_extractor = nn.Sequential(
            *list(vgg.features.children())[:36]
        )

        # Freeze VGG weights
        for param in self.feature_extractor.parameters():
            param.requires_grad = False

        # Put VGG in evaluation mode
        self.feature_extractor.eval()

        # ImageNet normalization values
        self.register_buffer(
            "mean",
            torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )

        self.register_buffer(
            "std",
            torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

    def forward(self, image):
        """
        Input:
            image : (B,3,H,W)

        Output:
            VGG feature maps
        """

        # Convert image from [-1, 1] to [0, 1]
        image = (image + 1.0) / 2.0

        # Normalize using ImageNet statistics
        image = (image - self.mean) / self.std

        return self.feature_extractor(image)

## Content (Perceptual) Loss
Comparing feature maps directly using VGG FE instead of comparing pixel-by-pixel.

In [ ]:
class ContentLoss(nn.Module):
    """
    Computes perceptual loss using VGG19 feature maps.
    """

    def __init__(self):
        super(ContentLoss, self).__init__()

        self.vgg = VGGFeatureExtractor()

        # Mean Squared Error on feature maps
        self.mse = nn.MSELoss()

    def forward(self, sr_image, hr_image):
        """
        Inputs:
            sr_image : Super-resolved image
            hr_image : Ground truth high-resolution image

        Returns:
            Content loss
        """
        # Extract feature maps
        sr_features = self.vgg(sr_image)

        # HR features should not backpropagate gradients
        with torch.no_grad():
            hr_features = self.vgg(hr_image)

        return self.mse(sr_features, hr_features)

## Adversarial Loss (Generator)
The Generator wants discriminator to predict 1 for fake images.

In [ ]:
class GeneratorLoss(nn.Module):
    """
    Computes adversarial loss for the generator.
    """

    def __init__(self):
        super(GeneratorLoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, fake_predictions):
        """
        fake_predictions:
        Output of discriminator for generated images.
        """
        target = torch.ones_like(fake_predictions)
        return self.bce(fake_predictions, target)

## Discriminator Loss
The discriminator learns to classify:

1 -> Real images

0 -> fake images

Combines both losses as both of them needs to be optimized, not required separately.

In [ ]:
class DiscriminatorLoss(nn.Module):
    """
    Computes discriminator loss.
    """

    def __init__(self):
        super(DiscriminatorLoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, real_predictions, fake_predictions):
        real_labels = torch.ones_like(real_predictions)
        fake_labels = torch.zeros_like(fake_predictions)

        real_loss = self.bce(real_predictions, real_labels)
        fake_loss = self.bce(fake_predictions, fake_labels)

        return real_loss + fake_loss

## Reconstruction (L1) and SRGAN Loss

In [ ]:
class ReconstructionLoss(nn.Module):
    """
    Used in both phases:
    content loss + pixel L1 loss
    """

    def __init__(self, content_weight=1.0, l1_weight=1.0):
        super().__init__()
        self.content_loss = ContentLoss()
        self.l1_loss = nn.L1Loss()
        self.content_weight = content_weight
        self.l1_weight = l1_weight

    def forward(self, sr_image, hr_image):
        content = self.content_loss(sr_image, hr_image)
        pixel_l1 = self.l1_loss(sr_image, hr_image)

        total = (
            self.content_weight * content
            + self.l1_weight * pixel_l1
        )
        return total, content, pixel_l1


class SRGANLoss(nn.Module):
    """
    GAN phase:
    content loss + L1 loss + adversarial loss
    """

    def __init__(self, reconstruction_loss, adv_weight=1e-3):
        super().__init__()
        self.reconstruction_loss = reconstruction_loss
        self.generator_loss = GeneratorLoss()
        self.adv_weight = adv_weight

    def forward(self, sr_image, hr_image, fake_predictions):
        reconstruction, content, pixel_l1 = self.reconstruction_loss(
            sr_image, hr_image
        )
        adversarial = self.generator_loss(fake_predictions)

        total = reconstruction + self.adv_weight * adversarial

        return total, content, pixel_l1, adversarial

## Initialize losses

In [ ]:
reconstruction_loss = ReconstructionLoss(
    content_weight=1.0,
    l1_weight=1.0
).to(device)

discriminator_loss = DiscriminatorLoss().to(device)

srgan_loss = SRGANLoss(
    reconstruction_loss=reconstruction_loss,
    adv_weight=1e-4
).to(device)

# Phase 5 - Training and Validation

## Optimizers
Separate optimizers for generator and discriminator

In [ ]:
generator_optimizer = torch.optim.Adam(
    generator.parameters(),
    lr=1e-4,
    betas=(0.9, 0.999) #(momentum,RMS), 90% of prev direction
)

discriminator_optimizer = torch.optim.Adam(
    discriminator.parameters(),
    lr=1e-4,
    betas=(0.9, 0.999)
)

## Learning Rate Scheduler and Epochs
Reducing LR after many epochs and setting number of epochs.

In [ ]:
generator_scheduler = torch.optim.lr_scheduler.StepLR(
    generator_optimizer,
    step_size=100,
    gamma=0.5
)

discriminator_scheduler = torch.optim.lr_scheduler.StepLR(
    discriminator_optimizer,
    step_size=100,
    gamma=0.5
)

# ---------------------------------------------------------------

PRETRAIN_EPOCHS = 75
GAN_EPOCHS = 125

TOTAL_EPOCHS = PRETRAIN_EPOCHS + GAN_EPOCHS

LATEST_CHECKPOINT_NAME = "latest_two_stage_checkpoint.pth"
CHECKPOINT_EVERY = 10

## Recording Losses
Storing losses to plot them later

In [ ]:
from torchmetrics.image import PeakSignalNoiseRatio
from torchmetrics.image import StructuralSimilarityIndexMeasure

# data range is 0 to 1 as normalized images are used
psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(device)
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

# metrics
generator_losses = []
discriminator_losses = []
validation_losses = []

content_losses = []
l1_losses = []
adversarial_losses = []
training_stages = []

psnr_scores = []
ssim_scores = []

## Training loop

In [ ]:
# make 'checkpoint' directory
import os

CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# Leave as None for a new run.
# To resume, set the path to a checkpoint created in the save cell below.
RESUME_CHECKPOINT_PATH = "/kaggle/input/models/abdulwahib480/image-res/pytorch/default/20/resume_checkpoint_two_stage.pth"

#best_validation_loss = float("inf")
#start_epoch = 0

if RESUME_CHECKPOINT_PATH is not None:
    checkpoint = torch.load(RESUME_CHECKPOINT_PATH, map_location=device)

    generator.load_state_dict(checkpoint["generator_state_dict"])
    discriminator.load_state_dict(checkpoint["discriminator_state_dict"])

    generator_optimizer.load_state_dict(
        checkpoint["generator_optimizer_state_dict"]
    )
    discriminator_optimizer.load_state_dict(
        checkpoint["discriminator_optimizer_state_dict"]
    )

    generator_scheduler.load_state_dict(
        checkpoint["generator_scheduler_state_dict"]
    )
    discriminator_scheduler.load_state_dict(
        checkpoint["discriminator_scheduler_state_dict"]
    )

    # next epoch to run.
    start_epoch = checkpoint["epoch"]

    saved_pretrain_epochs = checkpoint.get(
        "pretrain_epochs", PRETRAIN_EPOCHS
    )
    saved_gan_epochs = checkpoint.get("gan_epochs", GAN_EPOCHS)

    if (
        saved_pretrain_epochs != PRETRAIN_EPOCHS
        or saved_gan_epochs != GAN_EPOCHS
    ):
        raise ValueError(
            "Checkpoint phase lengths do not match the current "
            "PRETRAIN_EPOCHS and GAN_EPOCHS configuration."
        )

    best_validation_loss = checkpoint.get(
        "best_validation_loss", float("inf")
    )

    generator_losses = checkpoint.get("generator_losses", [])
    discriminator_losses = checkpoint.get("discriminator_losses", [])
    validation_losses = checkpoint.get("validation_losses", [])
    content_losses = checkpoint.get("content_losses", [])
    l1_losses = checkpoint.get("l1_losses", [])
    adversarial_losses = checkpoint.get("adversarial_losses", [])
    training_stages = checkpoint.get("training_stages", [])
    psnr_scores = checkpoint.get("psnr_scores", [])
    ssim_scores = checkpoint.get("ssim_scores", [])

    phase = (
        "pretraining"
        if start_epoch < PRETRAIN_EPOCHS
        else "GAN fine-tuning"
    )
    print(f"Resuming from epoch {start_epoch + 1} in {phase}.")
else:
    print("Starting new two-stage training run.")

In [ ]:
# `next_epoch` is used by the separate saving cell below this loop.
# It stores the next epoch to run when resuming.
next_epoch = start_epoch

for epoch in range(start_epoch, TOTAL_EPOCHS):
    generator.train()
    discriminator.train()

    is_pretraining = epoch < PRETRAIN_EPOCHS
    stage_name = "pretrain" if is_pretraining else "gan"

    running_generator_loss = 0.0
    running_discriminator_loss = 0.0
    running_content_loss = 0.0
    running_l1_loss = 0.0
    running_adversarial_loss = 0.0

    # =====================================================
    # Training
    # =====================================================
    for lr_images, hr_images in train_loader:
        lr_images = lr_images.to(device, non_blocking=True)
        hr_images = hr_images.to(device, non_blocking=True)

        # =====================================================
        # Stage 1: Generator pretraining
        # Content loss + L1 loss only
        # =====================================================
        if is_pretraining:
            generator_optimizer.zero_grad(set_to_none=True)

            sr_images = generator(lr_images)

            g_loss, content_value, l1_value = reconstruction_loss(
                sr_images,
                hr_images
            )

            g_loss.backward()
            generator_optimizer.step()

            running_generator_loss += g_loss.item()
            running_content_loss += content_value.item()
            running_l1_loss += l1_value.item()

        # =====================================================
        # Stage 2: GAN fine-tuning
        # Content loss + L1 loss + adversarial loss
        # =====================================================
        else:
            sr_images = generator(lr_images)

            # -------------------------
            # Train discriminator
            # -------------------------
            discriminator_optimizer.zero_grad(set_to_none=True)

            real_predictions = discriminator(hr_images)
            fake_predictions = discriminator(sr_images.detach())

            d_loss = discriminator_loss(
                real_predictions,
                fake_predictions
            )

            d_loss.backward()
            discriminator_optimizer.step()

            # -------------------------
            # Train generator
            # -------------------------
            # Freeze D parameters, but keep gradients flowing
            # through D back to the generator.
            for parameter in discriminator.parameters():
                parameter.requires_grad_(False)

            generator_optimizer.zero_grad(set_to_none=True)

            fake_predictions = discriminator(sr_images)

            g_loss, content_value, l1_value, adversarial_value = srgan_loss(
                sr_images,
                hr_images,
                fake_predictions
            )

            g_loss.backward()
            generator_optimizer.step()

            # Re-enable D gradients for the next batch.
            for parameter in discriminator.parameters():
                parameter.requires_grad_(True)

            running_generator_loss += g_loss.item()
            running_discriminator_loss += d_loss.item()
            running_content_loss += content_value.item()
            running_l1_loss += l1_value.item()
            running_adversarial_loss += adversarial_value.item()

    # =====================================================
    # Average training losses
    # =====================================================
    avg_generator_loss = running_generator_loss / len(train_loader)
    avg_content_loss = running_content_loss / len(train_loader)
    avg_l1_loss = running_l1_loss / len(train_loader)

    avg_discriminator_loss = (
        float("nan")
        if is_pretraining
        else running_discriminator_loss / len(train_loader)
    )

    avg_adversarial_loss = (
        float("nan")
        if is_pretraining
        else running_adversarial_loss / len(train_loader)
    )

    generator_losses.append(avg_generator_loss)
    discriminator_losses.append(avg_discriminator_loss)
    content_losses.append(avg_content_loss)
    l1_losses.append(avg_l1_loss)
    adversarial_losses.append(avg_adversarial_loss)
    training_stages.append(stage_name)

    # =====================================================
    # Validation
    # =====================================================
    generator.eval()

    running_validation_loss = 0.0
    running_psnr = 0.0
    running_ssim = 0.0

    with torch.no_grad():
        for lr_images, hr_images in val_loader:
            lr_images = lr_images.to(device, non_blocking=True)
            hr_images = hr_images.to(device, non_blocking=True)

            sr_images = generator(lr_images)

            # Validation always uses content + L1.
            validation_loss, _, _ = reconstruction_loss(
                sr_images,
                hr_images
            )
            running_validation_loss += validation_loss.item()

            # Convert from [-1, 1] to [0, 1] for image metrics.
            sr_images_01 = denormalize(sr_images).clamp(0.0, 1.0)
            hr_images_01 = denormalize(hr_images).clamp(0.0, 1.0)

            psnr = psnr_metric(sr_images_01, hr_images_01)
            ssim = ssim_metric(sr_images_01, hr_images_01)

            running_psnr += psnr.item()
            running_ssim += ssim.item()

    avg_validation_loss = running_validation_loss / len(val_loader)
    avg_psnr = running_psnr / len(val_loader)
    avg_ssim = running_ssim / len(val_loader)

    validation_losses.append(avg_validation_loss)
    psnr_scores.append(avg_psnr)
    ssim_scores.append(avg_ssim)

    # =====================================================
    # Save best generator
    # =====================================================
    if avg_validation_loss < best_validation_loss:
        best_validation_loss = avg_validation_loss

        best_name = (
            "best_generator_pretrain.pth"
            if is_pretraining
            else "best_generator_gan.pth"
        )

        torch.save(
            generator.state_dict(),
            os.path.join(CHECKPOINT_DIR, best_name)
        )

        print(f"Best {stage_name} generator saved.")

    # =====================================================
    # Learning-rate schedulers
    # =====================================================
    generator_scheduler.step()

    # Discriminator scheduler starts only in GAN phase.
    if not is_pretraining:
        discriminator_scheduler.step()

    # The separate saving cell uses this value.
    next_epoch = epoch + 1

    # =====================================================
    # Progress output
    # =====================================================
    if is_pretraining:
        print(
            f"Pretrain Epoch [{epoch + 1}/{PRETRAIN_EPOCHS}]\n"
            f"G Loss: {avg_generator_loss:.4f}\n"
            f"Content Loss: {avg_content_loss:.4f}\n"
            f"L1 Loss: {avg_l1_loss:.4f}\n"
            f"Valid Loss: {avg_validation_loss:.4f}\n"
            f"PSNR: {avg_psnr:.2f}\n"
            f"SSIM: {avg_ssim:.4f}\n"
        )
    else:
        gan_epoch = epoch - PRETRAIN_EPOCHS + 1

        print(
            f"GAN Epoch [{gan_epoch}/{GAN_EPOCHS}]\n"
            f"G Loss: {avg_generator_loss:.4f}\n"
            f"D Loss: {avg_discriminator_loss:.4f}\n"
            f"Content Loss: {avg_content_loss:.4f}\n"
            f"L1 Loss: {avg_l1_loss:.4f}\n"
            f"Adv Loss: {avg_adversarial_loss:.4f}\n"
            f"Valid Loss: {avg_validation_loss:.4f}\n"
            f"PSNR: {avg_psnr:.2f}\n"
            f"SSIM: {avg_ssim:.4f}\n"
        )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
RESUME_DIR = "/kaggle/working/Resume"
os.makedirs(RESUME_DIR, exist_ok=True)

resume_checkpoint = {
    "epoch": next_epoch,
    "pretrain_epochs": PRETRAIN_EPOCHS,
    "gan_epochs": GAN_EPOCHS,

    "generator_state_dict": generator.state_dict(),
    "discriminator_state_dict": discriminator.state_dict(),

    "generator_optimizer_state_dict": generator_optimizer.state_dict(),
    "discriminator_optimizer_state_dict": discriminator_optimizer.state_dict(),

    "generator_scheduler_state_dict": generator_scheduler.state_dict(),
    "discriminator_scheduler_state_dict": discriminator_scheduler.state_dict(),

    "best_validation_loss": best_validation_loss,

    "generator_losses": generator_losses,
    "discriminator_losses": discriminator_losses,
    "validation_losses": validation_losses,
    "content_losses": content_losses,
    "l1_losses": l1_losses,
    "adversarial_losses": adversarial_losses,
    "training_stages": training_stages,
    "psnr_scores": psnr_scores,
    "ssim_scores": ssim_scores,
}

latest_checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    LATEST_CHECKPOINT_NAME
)

resume_checkpoint_path = os.path.join(
    RESUME_DIR,
    "resume_checkpoint_two_stage.pth"
)

torch.save(resume_checkpoint, latest_checkpoint_path)
torch.save(resume_checkpoint, resume_checkpoint_path)

if next_epoch % CHECKPOINT_EVERY == 0:
    snapshot_path = os.path.join(
        CHECKPOINT_DIR,
        f"checkpoint_epoch_{next_epoch}.pth"
    )
    torch.save(resume_checkpoint, snapshot_path)

# Save generator exactly after the pretraining phase.
if next_epoch == PRETRAIN_EPOCHS:
    torch.save(
        generator.state_dict(),
        os.path.join(
            CHECKPOINT_DIR,
            "generator_pretrained_final.pth"
        )
    )

# save final models when all epochs finish.
if next_epoch >= TOTAL_EPOCHS:
    torch.save(
        generator.state_dict(),
        os.path.join(
            CHECKPOINT_DIR,
            "final_generator_two_stage.pth"
        )
    )
    torch.save(
        discriminator.state_dict(),
        os.path.join(
            CHECKPOINT_DIR,
            "final_discriminator_two_stage.pth"
        )
    )

print(
    f"Resume checkpoint saved for epoch {next_epoch + 1}: "
    f"{resume_checkpoint_path}"
)

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/resume_checkpoint",
    "zip",
    root_dir="/kaggle/working/Resume",
    base_dir="resume_checkpoint_two_stage.pth"
)

In [ ]:
# checking for best_ggenerator model

import os

path = "/kaggle/working/checkpoints/best_generator_gan.pth"

print("Exists:", os.path.exists(path))
if os.path.exists(path):
    print(f"Size: {os.path.getsize(path)/1024/1024:.2f} MB")

In [ ]:
# saving best generator model

import zipfile

zip_path = "/kaggle/working/best_generator_gan.zip"
model_path = "/kaggle/working/checkpoints/best_generator_gan.pth"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(model_path, arcname="best_generator.pth")

print("Done!")

## Plotting Metrics

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 10))

# Generator & Discriminator Loss
plt.subplot(2, 2, 1)
plt.plot(generator_losses, label="Generator")
plt.plot(discriminator_losses, label="Discriminator")
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

# Validation Loss
plt.subplot(2, 2, 2)
plt.plot(validation_losses)
plt.title("Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)

# PSNR
plt.subplot(2, 2, 3)
plt.plot(psnr_scores)
plt.title("PSNR")
plt.xlabel("Epoch")
plt.ylabel("dB")
plt.grid(True)

# SSIM
plt.subplot(2, 2, 4)
plt.plot(ssim_scores)
plt.title("SSIM")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.grid(True)

plt.tight_layout()
plt.show()

# Phase 6 - Inference and Testing

In [ ]:
import torch
from PIL import Image
from torchvision import transforms
from torchvision.transforms.functional import to_pil_image

# ======================================================
# Configuration
# ======================================================
MODEL_PATH = "/kaggle/input/models/abdulwahib480/image-res/pytorch/default/21/best_generator.pth"
INPUT_IMAGE = "/kaggle/input/datasets/noobyogi0100/low-resolution-photographs/dataset/img188.jpg"
OUTPUT_IMAGE = "/kaggle/working/enhanced.png"
SCALE = 2  # 2, 4 or 8
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================================================
# Load Generator
# ======================================================
generator = Generator().to(device)

generator.load_state_dict(
    torch.load(MODEL_PATH, map_location=device)
)
generator.eval()

# ======================================================
# Image Transform
# ======================================================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

# ======================================================
# Load Image
# ======================================================
image = Image.open(INPUT_IMAGE).convert("RGB")
input_tensor = transform(image).unsqueeze(0).to(device)

# ======================================================
# Determine number of passes
# ======================================================
if SCALE == 2:
    passes = 1
elif SCALE == 4:
    passes = 2
elif SCALE == 8:
    passes = 3
else:
    raise ValueError("SCALE must be 2, 4 or 8.")

# ======================================================
# Super Resolution
# ======================================================
with torch.no_grad():
    output = input_tensor
    for _ in range(passes):
        output = generator(output)

# ======================================================
# Convert back to image
# ======================================================
output = output.squeeze(0).cpu()

# Undo normalization
output = output * 0.5 + 0.5
output = torch.clamp(output, 0, 1)

output_image = to_pil_image(output)

# ======================================================
# Optional sharpening
# ======================================================
# from PIL import ImageFilter
# output_image = output_image.filter(
#     ImageFilter.UnsharpMask(radius=1.2, percent=120, threshold=3)
# )

# ======================================================
# saving high res image
# ======================================================
output_image.save(OUTPUT_IMAGE)
print("Inference completed successfully.")
print(f"Input Size: {image.size}")
print(f"Output Size: {output_image.size}")
print(f"Saved to: {OUTPUT_IMAGE}")

# Phase 7 - Fine-tuning For Other Image Degradations

## Mixed Degradation Dataset
This dataset implements the specific ratios: 40% Bicubic, 30% Blur, 20% Noise, and 10% Mixed (Blur+Noise+JPEG).

In [ ]:
import cv2
import io

class MixedDegradationDataset(SuperResolutionDataset):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def apply_degradations(self, hr_img_pil):
        # Convert to numpy for OpenCV processing
        hr_np = np.array(hr_img_pil)
        p = random.random()

        # 40% bicubic (handled by default logic below)
        # 30% blur + 2x downsampling
        # 20% noise + 2x downsampling
        # 10% blur + noise + JPEG + 2x downsampling

        temp_hr = hr_np.copy()

        # Apply Blur (Before downsampling)
        if p > 0.4: # Apply blur to 30% + 10% cases
            if p > 0.7 or p < 0.8: # Blur specific logic
                sigma = random.uniform(0.2, 1.5)
                temp_hr = cv2.GaussianBlur(temp_hr, (0, 0), sigma)

        # Downsample
        lr_img = Image.fromarray(temp_hr).resize(
            (self.hr_size // self.scale, self.hr_size // self.scale),
            resample=Image.BICUBIC
        )

        # Apply Noise and JPEG (After downsampling)
        lr_np = np.array(lr_img).astype(np.float32) / 255.0

        if 0.7 <= p < 0.9 or p >= 0.9: # Noise for 20% + 10% cases
            noise_std = random.uniform(0, 0.02)
            noise = np.random.normal(0, noise_std, lr_np.shape)
            lr_np = np.clip(lr_np + noise, 0, 1)

        lr_img = Image.fromarray((lr_np * 255).astype(np.uint8))

        if p >= 0.9: # JPEG for 10% case
            quality = random.randint(60, 100)
            buffer = io.BytesIO()
            lr_img.save(buffer, format="JPEG", quality=quality)
            lr_img = Image.open(buffer)

        return lr_img

    def preprocess(self, image):
        # Standard HR cropping/augmentation from parent
        if self.training:
            if image.width < self.hr_size or image.height < self.hr_size:
                image = TF.resize(image, (max(self.hr_size, image.height), max(self.hr_size, image.width)), interpolation=InterpolationMode.BICUBIC)
            i, j, h, w = transforms.RandomCrop.get_params(image, output_size=(self.hr_size, self.hr_size))
            hr = TF.crop(image, i, j, h, w)
            if random.random() < 0.5: hr = TF.hflip(hr)
        else:
            width = image.width - (image.width % self.scale)
            height = image.height - (image.height % self.scale)
            hr = TF.crop(image, 0, 0, height, width)

        # Apply custom degradation logic
        if self.training:
            lr = self.apply_degradations(hr)
        else:
            # For validation, we'll handle two sets outside
            lr = TF.resize(hr, (hr.height // self.scale, hr.width // self.scale), interpolation=InterpolationMode.BICUBIC)

        hr = TF.normalize(TF.to_tensor(hr), self.mean, self.std)
        lr = TF.normalize(TF.to_tensor(lr), self.mean, self.std)
        return lr, hr

## Setting DataLoaders

In [ ]:
train_mixed_dataset = MixedDegradationDataset(train_image_paths, training=True)
train_mixed_loader = DataLoader(train_mixed_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

# Dual Validation
val_clean_dataset = SuperResolutionDataset(test_image_paths, training=False)
val_degraded_dataset = MixedDegradationDataset(test_image_paths, training=False) # Note: For fixed validation, one would usually pre-generate these

val_clean_loader = DataLoader(val_clean_dataset, batch_size=1, shuffle=False)
val_degraded_loader = DataLoader(val_degraded_dataset, batch_size=1, shuffle=False)

## Fine-tuning Training Loop
Starting with Reconstruction-only for 20 epochs, followed by GAN for 20 epochs.

### Resuming Training

In [ ]:
import os
import torch

# Path to the fine-tuning checkpoint you want to resume from
# Set to None for a new fine-tuning run
RESUME_FT_CHECKPOINT_PATH = None

start_epoch_ft = 0
best_ft_val_loss = float('inf')

# Initialize metrics dictionary for new runs
ft_metrics = {
    'g_loss': [], 'd_loss': [], 
    'content_loss': [], 'l1_loss': [], 'adv_loss': [],
    'clean_psnr': [], 'clean_ssim': [],
    'deg_psnr': [], 'deg_ssim': []
}

if RESUME_FT_CHECKPOINT_PATH is not None:
    print(f"Resuming Fine-tuning from {RESUME_FT_CHECKPOINT_PATH}")
    checkpoint_ft = torch.load(RESUME_FT_CHECKPOINT_PATH, map_location=device)

    generator.load_state_dict(checkpoint_ft['generator_state_dict'])
    discriminator.load_state_dict(checkpoint_ft['discriminator_state_dict'])
    generator_optimizer.load_state_dict(checkpoint_ft['generator_optimizer_state_dict'])
    discriminator_optimizer.load_state_dict(checkpoint_ft['discriminator_optimizer_state_dict'])
    
    # Resume Schedulers
    if 'generator_scheduler_state_dict' in checkpoint_ft:
        generator_scheduler.load_state_dict(checkpoint_ft['generator_scheduler_state_dict'])
    if 'discriminator_scheduler_state_dict' in checkpoint_ft:
        discriminator_scheduler.load_state_dict(checkpoint_ft['discriminator_scheduler_state_dict'])

    start_epoch_ft = checkpoint_ft['epoch']
    best_ft_val_loss = checkpoint_ft.get('best_ft_val_loss', float('inf'))
    
    if 'ft_metrics' in checkpoint_ft:
        ft_metrics = checkpoint_ft['ft_metrics']
        
    print(f"Resumed fine-tuning from epoch {start_epoch_ft + 1}.")
    print(f"Best validation loss: {best_ft_val_loss:.6f}")
else:
    print("Starting new Phase 7 fine-tuning run.")

### Training loop

In [ ]:
# make 'checkpoint' directory
import os
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# config
FT_RECON_EPOCHS = 25
FT_GAN_EPOCHS = 50
FT_TOTAL_EPOCHS = FT_RECON_EPOCHS + FT_GAN_EPOCHS
PHASE5_EPOCHS = 200

# loading generator
BEST_P5_GEN = "/kaggle/input/models/abdulwahib480/image-res/pytorch/default/21/best_generator.pth"
if RESUME_FT_CHECKPOINT_PATH is None:
    generator.load_state_dict(torch.load(BEST_P5_GEN, map_location=device))
    print("Loaded Phase 5 generator.")

# initialized LR
if RESUME_FT_CHECKPOINT_PATH is None:
    for g in generator_optimizer.param_groups:
        g["lr"] = 5e-5
    for g in discriminator_optimizer.param_groups:
        g["lr"] = 5e-5
    print("Initialized fine-tuning learning rates.")

# We reset schedulers to match the new LR base if starting fresh
# (or they get loaded in the resume cell)

for epoch in range(start_epoch_ft, FT_TOTAL_EPOCHS):
    is_gan_phase = epoch >= FT_RECON_EPOCHS
    generator.train()
    if is_gan_phase:
        discriminator.train()
    else:
        discriminator.eval()

    running_g_loss = 0.0
    running_d_loss = 0.0
    running_content_loss = 0.0
    running_l1_loss = 0.0
    running_adv_loss = 0.0

    for lr_imgs, hr_imgs in train_mixed_loader:
        lr_imgs, hr_imgs = lr_imgs.to(device), hr_imgs.to(device)

        generator_optimizer.zero_grad()
        sr_imgs = generator(lr_imgs)

        if not is_gan_phase:
            g_loss, c_loss, l1_loss = reconstruction_loss(sr_imgs, hr_imgs)
            running_content_loss += c_loss.item()
            running_l1_loss += l1_loss.item()
            adv_val = 0.0
        else:
            discriminator_optimizer.zero_grad()
            d_loss = discriminator_loss(discriminator(hr_imgs), discriminator(sr_imgs.detach()))
            d_loss.backward()
            discriminator_optimizer.step()
            running_d_loss += d_loss.item()

            fake_preds = discriminator(sr_imgs)
            g_loss, c_loss, l1_loss, adv_loss = srgan_loss(sr_imgs, hr_imgs, fake_preds)
            running_content_loss += c_loss.item()
            running_l1_loss += l1_loss.item()
            adv_val = adv_loss.item()
            running_adv_loss += adv_val

        g_loss.backward()
        generator_optimizer.step()
        running_g_loss += g_loss.item()

    # Step Schedulers
    generator_scheduler.step()
    if is_gan_phase: discriminator_scheduler.step()

    # Validation
    generator.eval()
    val_metrics = {'clean': {'psnr': 0.0, 'ssim': 0.0}, 'degraded': {'psnr': 0.0, 'ssim': 0.0, 'loss': 0.0}}
    
    with torch.no_grad():
        for lr, hr in val_clean_loader:
            sr = generator(lr.to(device))
            sr_01, hr_01 = denormalize(sr).clamp(0,1), denormalize(hr.to(device)).clamp(0,1)
            val_metrics['clean']['psnr'] += psnr_metric(sr_01, hr_01).item()
            val_metrics['clean']['ssim'] += ssim_metric(sr_01, hr_01).item()

        for lr, hr in val_degraded_loader:
            hr_d = hr.to(device)
            sr = generator(lr.to(device))
            v_loss, _, _ = reconstruction_loss(sr, hr_d)
            val_metrics['degraded']['loss'] += v_loss.item()
            sr_01, hr_01 = denormalize(sr).clamp(0,1), denormalize(hr_d).clamp(0,1)
            val_metrics['degraded']['psnr'] += psnr_metric(sr_01, hr_01).item()
            val_metrics['degraded']['ssim'] += ssim_metric(sr_01, hr_01).item()

    n_train, n_clean, n_deg = len(train_mixed_loader), len(val_clean_loader), len(val_degraded_loader)
    
    # Update history
    ft_metrics['g_loss'].append(running_g_loss/n_train)
    ft_metrics['d_loss'].append(running_d_loss/n_train)
    ft_metrics['content_loss'].append(running_content_loss/n_train)
    ft_metrics['l1_loss'].append(running_l1_loss/n_train)
    ft_metrics['adv_loss'].append(running_adv_loss/n_train)
    ft_metrics['clean_psnr'].append(val_metrics['clean']['psnr']/n_clean)
    ft_metrics['clean_ssim'].append(val_metrics['clean']['ssim']/n_clean)
    ft_metrics['deg_psnr'].append(val_metrics['degraded']['psnr']/n_deg)
    ft_metrics['deg_ssim'].append(val_metrics['degraded']['ssim']/n_deg)

    global_epoch = PHASE5_EPOCHS + epoch + 1  #calculate epoch number
    print(f"\nFT Epoch [{global_epoch}/{PHASE5_EPOCHS + FT_TOTAL_EPOCHS}]")
    print(f"G-Loss: {ft_metrics['g_loss'][-1]:.4f}")
    print(f"D-Loss: {ft_metrics['d_loss'][-1]:.4f}")
    print(f"[Val Degraded]")
    print(f"Validation Loss: {avg_deg_loss:.6f}")
    print(f"PSNR: {ft_metrics['deg_psnr'][-1]:.2f}")
    print(f"SSIM: {ft_metrics['deg_ssim'][-1]:.4f}")

    avg_deg_loss = val_metrics['degraded']['loss']/n_deg
    if avg_deg_loss < best_ft_val_loss:
        best_ft_val_loss = avg_deg_loss
        torch.save(generator.state_dict(), os.path.join(CHECKPOINT_DIR, "best_generator_ft_mixed.pth"))
        print(f"New best generator saved!")

### Saving checkpoint

In [ ]:
# Cell for saving the full fine-tuning state
FT_RESUME_DIR = "/kaggle/working/Resume_FT"
os.makedirs(FT_RESUME_DIR, exist_ok=True)

ft_resume_checkpoint = {
    "epoch": epoch + 1,
    "generator_state_dict": generator.state_dict(),
    "discriminator_state_dict": discriminator.state_dict(),
    "generator_optimizer_state_dict": generator_optimizer.state_dict(),
    "discriminator_optimizer_state_dict": discriminator_optimizer.state_dict(),
    "generator_scheduler_state_dict": generator_scheduler.state_dict(),
    "discriminator_scheduler_state_dict": discriminator_scheduler.state_dict(),
    "best_ft_val_loss": best_ft_val_loss,
    "ft_metrics": ft_metrics
}

resume_ft_path = os.path.join(FT_RESUME_DIR, "resume_checkpoint_ft_mixed.pth")
torch.save(ft_resume_checkpoint, resume_ft_path)
print(f"Complete fine-tuning checkpoint saved at: {resume_ft_path}")